In [1]:
import pandas as pd
import yfinance as yf
import numpy as np
#import pandas_datareader.data as web
#import datetime
import os
#import matplotlib.pyplot as plt
#import plotly
import time
from datetime import datetime

#### 함수정의

In [2]:
def getDailyStockInfo(target, startDate, endDate):
    target = target.replace('.', '-')
    stStartDate = startDate.strftime('%Y-%m-%d')
    stEndDate = endDate.strftime('%Y-%m-%d')
    stockDf = yf.download([target], start=stStartDate, end=stEndDate,auto_adjust=False, progress=False)
    stockDf.columns = stockDf.columns.droplevel(1)  #header(Ticker) 삭제

    stockDf.rename (columns={'Adj Close':'adj_close'}, inplace=True)
    
    return stockDf

In [7]:
#현재 디렉토리 경로
dirNm = "dailyStock"
vspwd = os.getcwd()

#디렉토리 재설정
os.chdir(vspwd)

#일별 주가정보 디렉토리 생성
os.makedirs(dirNm, exist_ok=True)

companyDf =  pd.read_csv("companyList.csv", encoding="utf-8")

In [12]:
endDate = datetime.today().replace(
    hour=0,
    minute=0,
    second=0,
    microsecond=0
)

for row in companyDf.itertuples(): 
    startDate = datetime.strptime("2000-01-01", "%Y-%m-%d")
    
    target = row.Symbol
    file_path = f"{dirNm}/{target}.csv"

    read_df = None
    select_df = None
    
    #저장된 일변주가정보가 없을 경우
    if os.path.exists(file_path): 
        read_df =  pd.read_csv(file_path, encoding="utf-8")
        startDate = datetime.strptime(read_df['Date'].max(), "%Y-%m-%d")
        
        #조회해온 df와 구조가 같도록 Date를 인덱스로 설정
        read_df = read_df.set_index('Date')  

    
    #마지막으로 저장된 일별 주식정보가 최신이 아닐 경우
    if startDate < endDate:        
        select_df = getDailyStockInfo(target, startDate, endDate)    
    else:
        print(f"{target} complete => 조회시작일시:{startDate} 조회종료일시:{endDate}")
        continue

    #조회해 온 데이터가 없을 경우
    if select_df is None or select_df.empty: 
        print(f"{target} empty => 조회시작일시:{startDate} 조회종료일시:{endDate}")
        continue
    
    #저장 데이터와 조회 데이터 합치기
    read_df = pd.concat([read_df, select_df],ignore_index=False)

    #중복된 일자가 있을 경우 조회 데이터를 남김
    read_df = read_df[~read_df.index.duplicated(keep='last')]
    
    #인덱스를 컬럼으로 변환
    read_df = read_df.reset_index()
        
    # Date를 datetime으로 변환 후 YYYY-MM-DD 문자열로 변경
    read_df['Date'] = pd.to_datetime(read_df['Date']).dt.strftime('%Y-%m-%d')
        
    #csv로 저장
    read_df.to_csv(file_path, encoding='utf-8', index=False)

    time.sleep(2)
        
    print(f"{target} complete => 조회시작일시:{startDate} 조회종료일시:{endDate}")

^IRX complete => 조회시작일시:2000-01-01 00:00:00 조회종료일시:2026-09-17 00:00:00
